# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")

carpeta = Path("/content/drive/MyDrive/lecture6/challenge")

if not carpeta.is_dir():
    raise FileNotFoundError(f"No se encontró la carpeta: {carpeta}")

os.chdir(carpeta)

archivos_requeridos = [
    carpeta / "data" / "processed" / "train.csv",
    carpeta / "artifacts" / "data_contract.json"
]

for archivo in archivos_requeridos:
    if not archivo.is_file():
        raise FileNotFoundError(f"Falta el archivo: {archivo}")

print("Carpeta de trabajo:", Path.cwd())
print("Archivos del Challenge 1 encontrados.")

Mounted at /content/drive
Carpeta de trabajo: /content/drive/MyDrive/lecture6/challenge
Archivos del Challenge 1 encontrados.


In [2]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [3]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [4]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [5]:
feature_columns = contract["features"]
RANDOM_STATE = int(contract["random_state"])

# Árbol: imputación, sin escalado.
tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(strategy="median"),
            feature_columns
        )
    ],
    remainder="drop"
)

# Regresión logística y KNN: imputación y escalado.
scaled_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            feature_columns
        )
    ],
    remainder="drop"
)

print("Preprocesadores creados.")

Preprocesadores creados.


**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Escribe aquí tu respuesta.

**Respuesta:**

El StandardScaler debe estar dentro del pipeline para que, en cada partición de la validación cruzada, calcule la media y la desviación estándar únicamente con los datos usados para entrenar.

Si se ajustara antes de la validación cruzada utilizando todo el conjunto de entrenamiento, también incorporaría información de los ejemplos que después funcionan como validación. Esto produciría una fuga de información y podría generar una estimación demasiado optimista del desempeño.

El pipeline aplica a la partición de validación la transformación aprendida con la partición de entrenamiento. Además, permite guardar el preprocesamiento y el modelo juntos para utilizarlos posteriormente sobre nuevos datos.

## 3. Modelos candidatos

In [6]:
models = {
    "DecisionTree": Pipeline([
        ("preprocessor", clone(tree_preprocessor)),
        (
            "model",
            DecisionTreeClassifier(random_state=RANDOM_STATE)
        )
    ]),
    "LogisticRegression": Pipeline([
        ("preprocessor", clone(scaled_preprocessor)),
        (
            "model",
            LogisticRegression(
                max_iter=5000,
                random_state=RANDOM_STATE
            )
        )
    ]),
    "KNN": Pipeline([
        ("preprocessor", clone(scaled_preprocessor)),
        ("model", KNeighborsClassifier())
    ])
}

print("Modelos candidatos:")
for nombre in models:
    print("-", nombre)

Modelos candidatos:
- DecisionTree
- LogisticRegression
- KNN


## 4. Comparación inicial con validación cruzada

In [7]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

initial_rows = []

for nombre, pipeline in models.items():
    scores = cross_validate(
        estimator=pipeline,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring="accuracy",
        return_train_score=True,
        n_jobs=-1,
        error_score="raise"
    )

    train_mean = scores["train_score"].mean()
    validation_mean = scores["test_score"].mean()

    initial_rows.append({
        "model": nombre,
        "train_accuracy_mean": train_mean,
        "validation_accuracy_mean": validation_mean,
        "validation_accuracy_std": scores["test_score"].std(),
        "train_validation_gap": train_mean - validation_mean
    })

initial_comparison = (
    pd.DataFrame(initial_rows)
    .sort_values(
        "validation_accuracy_mean",
        ascending=False
    )
    .reset_index(drop=True)
)

display(initial_comparison.round(4))

,model,train_accuracy_mean,validation_accuracy_mean,validation_accuracy_std,train_validation_gap
0,LogisticRegression,1.0000,0.9791,0.0277,0.0209
1,KNN,0.9771,0.9507,0.0353,0.0264
2,DecisionTree,1.0000,0.9022,0.0586,0.0978


## 5. Optimización de hiperparámetros

In [8]:
param_grids = {
    "DecisionTree": {
        "model__max_depth": [2, 3, 5, None],
        "model__min_samples_leaf": [1, 2, 4]
    },
    "LogisticRegression": {
        "model__C": [0.01, 0.1, 1.0, 10.0]
    },
    "KNN": {
        "model__n_neighbors": [3, 5, 7, 9],
        "model__weights": ["uniform", "distance"]
    }
}

searches = {}
tuned_rows = []

for nombre, pipeline in models.items():
    search = GridSearchCV(
        estimator=clone(pipeline),
        param_grid=param_grids[nombre],
        scoring="accuracy",
        cv=cv,
        refit=True,
        return_train_score=True,
        n_jobs=-1,
        error_score="raise"
    )

    search.fit(X_train, y_train)
    searches[nombre] = search

    best_index = search.best_index_
    results = search.cv_results_

    train_mean = float(
        results["mean_train_score"][best_index]
    )
    validation_mean = float(search.best_score_)

    tuned_rows.append({
        "model": nombre,
        "train_accuracy_mean": train_mean,
        "validation_accuracy_mean": validation_mean,
        "validation_accuracy_std": float(
            results["std_test_score"][best_index]
        ),
        "train_validation_gap": train_mean - validation_mean,
        "best_params": search.best_params_
    })

tuned_comparison = (
    pd.DataFrame(tuned_rows)
    .sort_values(
        "validation_accuracy_mean",
        ascending=False
    )
    .reset_index(drop=True)
)

display(tuned_comparison.round(4))

,model,train_accuracy_mean,validation_accuracy_mean,validation_accuracy_std,train_validation_gap,best_params
0,LogisticRegression,1.0000,0.9791,0.0277,0.0209,{'model__C': 1.0}
1,KNN,0.9648,0.9579,0.0405,0.0069,"{'model__n_neighbors': 7, 'model__weights': 'u..."
2,DecisionTree,0.9859,0.9025,0.0762,0.0835,"{'model__max_depth': 5, 'model__min_samples_le..."


## 6. Selección y persistencia del campeón

In [9]:
import sklearn

# Selección según la mayor accuracy promedio de CV.
champion_name = max(
    searches,
    key=lambda nombre: searches[nombre].best_score_
)

champion_search = searches[champion_name]
champion = champion_search.best_estimator_

best_index = champion_search.best_index_
results = champion_search.cv_results_

train_accuracy = float(
    results["mean_train_score"][best_index]
)
validation_accuracy = float(champion_search.best_score_)
validation_std = float(
    results["std_test_score"][best_index]
)

# Guardar el pipeline completo.
joblib.dump(
    champion,
    ARTIFACTS_DIR / "champion_model.joblib"
)

# Guardar los resultados de todas las búsquedas.
all_cv_results = []

for nombre, search in searches.items():
    model_results = pd.DataFrame(search.cv_results_)
    model_results.insert(0, "model", nombre)
    all_cv_results.append(model_results)

cv_results = pd.concat(
    all_cv_results,
    ignore_index=True
)

cv_results.to_csv(
    REPORTS_DIR / "cv_results.csv",
    index=False
)

# Guardar metadatos.
training_metadata = {
    "champion_name": champion_name,
    "best_params": champion_search.best_params_,
    "best_cv_score": validation_accuracy,
    "cv_accuracy_std": validation_std,
    "train_accuracy_mean": train_accuracy,
    "train_validation_gap": (
        train_accuracy - validation_accuracy
    ),
    "scoring": "accuracy",
    "cv": {
        "n_splits": 5,
        "shuffle": True,
        "random_state": RANDOM_STATE
    },
    "random_state": RANDOM_STATE,
    "target": contract["target"],
    "features": contract["features"],
    "n_training_samples": int(len(X_train)),
    "selection_used_test": False,
    "sklearn_version": sklearn.__version__
}

with (
    ARTIFACTS_DIR / "training_metadata.json"
).open("w", encoding="utf-8") as archivo:
    json.dump(
        training_metadata,
        archivo,
        indent=2,
        ensure_ascii=False
    )

print("Modelo campeón:", champion_name)
print("Mejores parámetros:", champion_search.best_params_)
print(f"Accuracy promedio de CV: {validation_accuracy:.4f}")
print(f"Desviación estándar de CV: {validation_std:.4f}")
print(f"Accuracy promedio de entrenamiento: {train_accuracy:.4f}")
print(
    "Brecha entrenamiento-validación:",
    f"{train_accuracy - validation_accuracy:.4f}"
)

print("\nArchivos guardados:")
print(ARTIFACTS_DIR / "champion_model.joblib")
print(ARTIFACTS_DIR / "training_metadata.json")
print(REPORTS_DIR / "cv_results.csv")

Modelo campeón: LogisticRegression
Mejores parámetros: {'model__C': 1.0}
Accuracy promedio de CV: 0.9791
Desviación estándar de CV: 0.0277
Accuracy promedio de entrenamiento: 1.0000
Brecha entrenamiento-validación: 0.0209

Archivos guardados:
/content/drive/MyDrive/lecture6/challenge/artifacts/champion_model.joblib
/content/drive/MyDrive/lecture6/challenge/artifacts/training_metadata.json
/content/drive/MyDrive/lecture6/challenge/reports/cv_results.csv


In [10]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

> Explica la decisión considerando desempeño, variabilidad, sobreajuste, complejidad e interpretabilidad.


**Justificación de la selección:**

Seleccioné **LogisticRegression**, con **C = 1.0**, porque obtuvo la mayor accuracy promedio en validación cruzada: **97.91 %**, frente al **95.79 % de KNN** y al **90.25 % de DecisionTree**.

También presentó la menor desviación estándar entre las cinco particiones: **0.0277**, frente a **0.0405 de KNN** y **0.0762 de DecisionTree**. Esto indica un desempeño más estable en las particiones evaluadas.

En cuanto al sobreajuste, LogisticRegression alcanzó una accuracy promedio de entrenamiento del **100 %**, con una brecha de **2.09 puntos porcentuales** respecto a validación. Esta diferencia es pequeña, aunque no permite descartar completamente el sobreajuste. KNN tuvo una brecha menor, de **0.69 puntos porcentuales**, pero también menor accuracy de validación. El árbol presentó la mayor brecha, de **8.35 puntos porcentuales**, lo que sugiere una mayor tendencia al sobreajuste.

Respecto a la complejidad, la regresión logística es un modelo lineal con regularización que permite realizar predicciones de forma eficiente. Sus coeficientes permiten analizar cómo cada característica influye en las puntuaciones de las clases, considerando la estandarización y la correlación entre variables. Aunque un árbol puede resultar más sencillo de visualizar, LogisticRegression ofreció el mejor equilibrio observado entre desempeño, estabilidad e interpretabilidad.

La selección se realizó exclusivamente mediante validación cruzada sobre entrenamiento, **sin utilizar el conjunto de prueba**. Guardé el pipeline completo, sus metadatos y los resultados de la búsqueda. La evaluación final en test permitirá estimar su desempeño sobre los datos reservados.

## Checklist

- [x] Todo el procesamiento está dentro de pipelines.
- [x] Comparé tres familias de modelos.
- [x] Optimicé hiperparámetros con CV.
- [x] Elegí el campeón sin usar test.
- [x] Guardé el pipeline completo y sus metadatos.